In [67]:
from local_tests.preprocess_bigdata_test import test_init_dataset ,test_tokenize

In [ ]:
from datasets import load_dataset, IterableDataset
from transformers import AutoTokenizer

# CHANGE RETURN TO TRUE only after you have confirmed the dataset is being streamed
# Otherwise your Gradescope submission may take a very long time to run or even timeout.
def run_autograder():
    return True

class Preprocess:
    def __init__(self):
        '''
        Initialize the dataset and the tokenizer. 

        For self.dataset, most of the code is given. We are importing the Wikipedia dataset from Hugging Face.
        However, because of the big data context (size of the dataset is 20GB!), we will need to alter it to 
        allow for streaming (https://huggingface.co/docs/datasets/stream) so that we don't load all of it into
        memory. Note that after allowing for streaming, self.dataset will be of the type IterableDataset, 
        documentation of which can be found here: 
        https://huggingface.co/docs/datasets/v2.14.4/en/package_reference/main_classes#datasets.IterableDataset.
        You could also try running the code once as is to compare streaming vs non-streaming, and see why it is 
        so important!

        For self.tokenizer, let's use the AutoTokenizer we imported from the Hugging Face transformers library. 
        We will use the 'distilbert-base-cased' tokenizer, which is a distilled version of the LLM, BERT. We will
        be learning more about the inner-workings of BERT later on in the course.

        Autotokenizer: https://huggingface.co/transformers/v3.0.2/model_doc/auto.html
        Distilbert: https://huggingface.co/distilbert-base-cased
        Tokenizer: https://huggingface.co/docs/transformers/main_classes/tokenizer
        '''
        #5.1.1
        self.dataset = load_dataset('wikimedia/wikipedia', name = "20231101.en", split='train',streaming=True)

        #5.1.2
        self.tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased',eos_token='[SEP]',padding_side='right',truncation_side='right')

    def tokenize(self, batch: dict[str, list[str]], max_length: int=100) -> dict[str, list[str]]:
        '''
        This is a helper method that will be called by preprocess_text() to tokenize the 'text' column in batch. 
        Keep in mind that in this method we are not just tokenizing the text using Distilbert, we are then converting 
        those numerical token ids back to English tokens. Documentation for tokenizers can be found here: 
        https://huggingface.co/docs/transformers/main_classes/tokenizer

        For the tokenizer, we always want the arrays of tokens to be size max_length. Make sure to set the padding and
        truncation strategies to allow for this.

        Args:
            batch: group of samples streamed from the dataset
            max_length: maximum length of the tokenized text to pad/truncate to
        Return:
            tokens_dict: dictionary mapping the word 'tokens' to a list of English tokens for each sample in the batch. 
                         This output will then be used in the preprocess_text() method to add a new 'tokens' column to 
                         self.dataset.
        '''
        # 5.1.2
        
        decode_list = []
        for e in batch['text']:
          
            econded = self.tokenizer.encode(e,max_length=max_length,truncation=True,padding='max_length',add_special_tokens=True)

            decode_list.append(self.tokenizer.convert_ids_to_tokens(econded))
        
        tokens_dict = {'tokens':decode_list}

        return tokens_dict


    
    def preprocess_text(self) -> IterableDataset:
        '''
        Here, we will apply the self.tokenize method on self.dataset. Remember to only keep the columns 'id', 'title', and 
        'tokens'. Make sure to use batching (use batch of 1000) to allow for proper processing speed. 

        The Dataset map function and its parameters may be very useful.

        Return:
            dataset_cleaned: Iterable Dataset with the 'id', 'title', and new 'tokens' column added
        '''
        #5.1.3

        datasets= self.dataset.map(self.tokenize,batch_size=1000,batched=True)

        dataset_cleaned = datasets.select_columns(['id', 'title','tokens'])

        return dataset_cleaned
    

    




In [81]:
test_class = Preprocess()

test_class.preprocess_text()

IterableDataset({
    features: Unknown,
    n_shards: 41
})

In [66]:
tokenizer_example_sentences_case1 = {"text": ["This sentence is the truth!!", "This sentence isn't the truth.", "One of these sentences isn't a truth?"]}
# Define the new sentence with more than 15 words
tokenizer_example_sentences_case2 = {"text": ["This sentence is the truth!!", "This sentence isn't the truth.", "One of these sentences isn't a truth?","During the early morning hours, the quiet town was full of activity as people prepared for the upcoming festival, filling the streets with vibrant colors and delightful scents."]}

In [83]:
t1= test_class.tokenize(tokenizer_example_sentences_case1,  max_length=15)
t2= test_class.tokenize(tokenizer_example_sentences_case2,  max_length=15)

In [84]:
test_tokenize(t1,t2)

Expected Output for case 1: {'tokens': [['[CLS]', 'This', 'sentence', 'is', 'the', 'truth', '!', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['[CLS]', 'This', 'sentence', 'isn', "'", 't', 'the', 'truth', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['[CLS]', 'One', 'of', 'these', 'sentences', 'isn', "'", 't', 'a', 'truth', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]']]}

Your Output for case 1: {'tokens': [['[CLS]', 'This', 'sentence', 'is', 'the', 'truth', '!', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['[CLS]', 'This', 'sentence', 'isn', "'", 't', 'the', 'truth', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['[CLS]', 'One', 'of', 'these', 'sentences', 'isn', "'", 't', 'a', 'truth', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]']]}

Expected Output for case 2: {'tokens': [['[CLS]', 'This', 'sentence', 'is', 'the', 'truth', '!', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'], ['[CLS]', 'This', 's

In [43]:
x=["This sentence is the truth!!", "This sentence isn't the truth.", "One of these sentences isn't a truth?","During the early morning hours, the quiet town was full of activity as people prepared for the upcoming festival, filling the streets with vibrant colors and delightful scents."]

In [52]:
for e in x:
    econded = test_class.tokenizer.encode(e,max_length=14,truncation=True,padding='max_length',add_special_tokens=True,split_special_tokens=True,eos_token='[SEP]',padding_side='right',truncation_side='right')

    print(test_class.tokenizer.convert_ids_to_tokens(econded))

['[CLS]', 'This', 'sentence', 'is', 'the', 'truth', '!', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['[CLS]', 'This', 'sentence', 'isn', "'", 't', 'the', 'truth', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['[CLS]', 'One', 'of', 'these', 'sentences', 'isn', "'", 't', 'a', 'truth', '?', '[SEP]', '[PAD]', '[PAD]']
['[CLS]', 'During', 'the', 'early', 'morning', 'hours', ',', 'the', 'quiet', 'town', 'was', 'full', 'of', '[SEP]']


In [53]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased')

In [54]:
def tokenize( batch: dict[str, list[str]], max_length: int=100) -> dict[str, list[str]]:
        '''
        This is a helper method that will be called by preprocess_text() to tokenize the 'text' column in batch. 
        Keep in mind that in this method we are not just tokenizing the text using Distilbert, we are then converting 
        those numerical token ids back to English tokens. Documentation for tokenizers can be found here: 
        https://huggingface.co/docs/transformers/main_classes/tokenizer

        For the tokenizer, we always want the arrays of tokens to be size max_length. Make sure to set the padding and
        truncation strategies to allow for this.

        Args:
            batch: group of samples streamed from the dataset
            max_length: maximum length of the tokenized text to pad/truncate to
        Return:
            tokens_dict: dictionary mapping the word 'tokens' to a list of English tokens for each sample in the batch. 
                         This output will then be used in the preprocess_text() method to add a new 'tokens' column to 
                         self.dataset.
        '''
        # 5.1.2
        
        decode_list = []
        for e in batch['text']:
          
            econded = tokenizer.encode(e,max_length=max_length,truncation=True,padding='max_length',add_special_tokens=True,split_special_tokens=True,eos_token='[SEP]',padding_side='right',truncation_side='right')

            decode_list.append(tokenizer.convert_ids_to_tokens(econded))
        
        tokens_dict = {'tokens':decode_list}

        return tokens_dict

In [12]:
test_text = next(iter(test_class.dataset))

In [ ]:
clean_ds = test_class.dataset.add_column('tokens',test_class.dataset.map(tokenize,batch_size=1000,))

In [63]:
z = test_class.dataset.map(tokenize,batch_size=1000)

z = z.select_columns(['id', 'title','tokens'])

In [64]:
next(iter(z))

{'id': '12',
 'title': 'Anarchism',
 'tokens': [['[CLS]',
   'A',
   '[SEP]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]'

In [59]:
next(iter(test_class.dataset))

{'id': '12',
 'url': 'https://en.wikipedia.org/wiki/Anarchism',
 'title': 'Anarchism',
 'text': 'Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).\n\nHumans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. Dur

In [28]:
econded = test_class.tokenizer.encode(test_text['text'],max_length=100,bos_token='[CLS]')

In [29]:
test_class.tokenizer.decode(econded).split()

['[CLS]',
 'Anarchism',
 'is',
 'a',
 'political',
 'philosophy',
 'and',
 'movement',
 'that',
 'is',
 'skeptical',
 'of',
 'all',
 'justifications',
 'for',
 'authority',
 'and',
 'seeks',
 'to',
 'abolish',
 'the',
 'institutions',
 'it',
 'claims',
 'maintain',
 'unnecessary',
 'coercion',
 'and',
 'hierarchy,',
 'typically',
 'including',
 'nation',
 '-',
 'states,',
 'and',
 'capitalism.',
 'Anarchism',
 'advocates',
 'for',
 'the',
 'replacement',
 'of',
 'the',
 'state',
 'with',
 'stateless',
 'societies',
 'and',
 'voluntary',
 'free',
 'associations.',
 'As',
 'a',
 'historically',
 'left',
 '-',
 'wing',
 'movement,',
 'this',
 'reading',
 'of',
 'anarchism',
 'is',
 'placed',
 'on',
 'the',
 'farthest',
 'left',
 'of',
 'the',
 'political',
 'spectrum,',
 'usually',
 'described',
 'as',
 'the',
 'libertarian',
 'wing',
 '[SEP]']

In [13]:

test_class.tokenizer(test_text['text'])

{'input_ids': [101, 9954, 10340, 1863, 1110, 170, 1741, 5027, 1105, 2230, 1115, 1110, 21333, 1104, 1155, 22647, 1116, 1111, 3748, 1105, 11053, 1106, 170, 15792, 2944, 1103, 4300, 1122, 3711, 4731, 14924, 1884, 1200, 16012, 1105, 14486, 117, 3417, 1259, 3790, 118, 2231, 117, 1105, 20582, 119, 9954, 10340, 1863, 15193, 1111, 1103, 5627, 1104, 1103, 1352, 1114, 1352, 2008, 9306, 1105, 12048, 1714, 9815, 119, 1249, 170, 8528, 1286, 118, 3092, 2230, 117, 1142, 3455, 1104, 1126, 1813, 25564, 1110, 1973, 1113, 1103, 1677, 22722, 1286, 1104, 1103, 1741, 10122, 117, 1932, 1758, 1112, 1103, 181, 24851, 24279, 3092, 1104, 1103, 11181, 2230, 113, 181, 24851, 24279, 20560, 114, 119, 22143, 1138, 2077, 1107, 9306, 1443, 4698, 20844, 5970, 10340, 1905, 1263, 1196, 1103, 4544, 1104, 2231, 117, 9695, 1116, 117, 1137, 8207, 1116, 119, 1556, 1103, 3606, 1104, 6967, 20844, 5970, 10340, 4571, 3470, 117, 188, 2093, 26265, 1755, 3748, 1145, 3152, 119, 1966, 10749, 1104, 23301, 4133, 1132, 1276, 1155, 2032, 1

In [13]:
dataset = load_dataset('wikimedia/wikipedia', name = "20231101.en", split='train')

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Generating train split: 100%|██████████| 6407814/6407814 [00:52<00:00, 121475.73 examples/s]


In [7]:
dataset.add_column(Preprocess())

NameError: name 'dataset' is not defined

In [12]:
t=AutoTokenizer.from_pretrained('distilbert-base-cased')

In [ ]:
t.

In [ ]:
t.add_tokens()

In [ ]:
tests = [['will','will'],['brennan','brennan']]